# YouTube AI Studio v7 - Generador Visual

Notebook automatizado para Kaggle (GPU T4 x2).

**Modelos:**
- **FLUX.1-schnell** para imagenes (4 steps, 1280x720)
- **Real-ESRGAN x2** para upscaling (imagenes a 1920x1080, video 480P a 720P)
- **Wan 2.1 I2V 14B 480P** para video IA (25 steps, 41 frames)

**Flujo:**
1. Lee `prompts.json` desde el dataset `ytcreator-prompts`
2. Genera imagenes con FLUX.1-schnell (1280x720)
3. Upscale imagenes con Real-ESRGAN (1920x1080)
4. Genera video con Wan 2.1 para escenas marcadas con `usa_video_ia`
5. Upscale videos con Real-ESRGAN (480P -> 720P)
6. Aplica zoom/pan cinematico a escenas sin video IA
7. Guarda todo en `/kaggle/working/` (output automatico)

In [ ]:
# Celda 1 - Instalar dependencias
!pip install -q diffusers==0.31.0 transformers~=4.44.0 accelerate safetensors
!pip install -q sentencepiece protobuf
!pip install -q opencv-python-headless pillow imageio imageio-ffmpeg
!pip install -q realesrgan basicsr
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth -P /kaggle/working/weights/

In [ ]:
# Celda 2 - Leer prompts.json desde el dataset de Kaggle
import json
from pathlib import Path

DATASET_PATH = Path("/kaggle/input/ytcreator-prompts/prompts.json")
OUTPUT_DIR = Path("/kaggle/working")

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"No se encontro {DATASET_PATH}. "
        "Asegurate de que el dataset 'ytcreator-prompts' este agregado como input."
    )

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

PROYECTO_ID = data["proyecto_id"]
ESTILO_APLICADO = data.get("estilo_aplicado", "sin_estilo")
ESCENAS = data["escenas"]

escenas_video = [e for e in ESCENAS if e.get("usa_video_ia", False)]
escenas_zoom = [e for e in ESCENAS if not e.get("usa_video_ia", False)]

print(f"Proyecto: {PROYECTO_ID}")
print(f"Estilo aplicado: {ESTILO_APLICADO}")
print(f"Total escenas: {len(ESCENAS)}")
print(f"  - Con video IA (Wan 2.1): {len(escenas_video)}")
print(f"  - Con zoom/pan: {len(escenas_zoom)}")
print(f"\nPrimer prompt: {ESCENAS[0]['prompt'][:120]}...")
if ESCENAS[0].get("negative_prompt"):
    print(f"Primer negative: {ESCENAS[0]['negative_prompt'][:80]}...")

In [ ]:
# Celda 3 - Cargar FLUX.1-schnell y generar imagenes (1280x720)
import torch
import time
from diffusers import FluxPipeline
from PIL import Image as PILImage

MODEL_ID = "black-forest-labs/FLUX.1-schnell"

pipe_img = FluxPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
)
pipe_img.enable_model_cpu_offload()

print(f"Modelo cargado: {MODEL_ID}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

WIDTH = 1280
HEIGHT = 720
STEPS = 4
SEED_BASE = 42

imagenes_generadas = {}

print(f"\nGenerando {len(ESCENAS)} imagenes con FLUX.1-schnell...\n")
t0 = time.time()

for i, escena in enumerate(ESCENAS):
    numero = escena["numero"]
    prompt = escena["prompt"]
    output_path = OUTPUT_DIR / f"escena_{numero:02d}.png"

    print(f"[{i+1}/{len(ESCENAS)}] Escena {numero}...")

    generator = torch.Generator("cpu").manual_seed(SEED_BASE + numero)

    image = pipe_img(
        prompt=prompt,
        guidance_scale=0.0,
        num_inference_steps=STEPS,
        max_sequence_length=256,
        generator=generator,
        width=WIDTH,
        height=HEIGHT,
    ).images[0]

    image.save(str(output_path))
    imagenes_generadas[numero] = str(output_path)
    elapsed = time.time() - t0
    per_img = elapsed / (i + 1)
    remaining = per_img * (len(ESCENAS) - i - 1)
    print(f"  OK ({per_img:.1f}s/img, ~{remaining/60:.1f}min restantes)")

    torch.cuda.empty_cache()

print(f"\n{len(imagenes_generadas)} imagenes generadas en {(time.time()-t0)/60:.1f} min")

In [ ]:
# Celda 4 - Real-ESRGAN: upscale imagenes de 1280x720 a 1920x1080
import cv2
import numpy as np
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

model_esrgan = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=2)
upsampler = RealESRGANer(
    scale=2,
    model_path="/kaggle/working/weights/RealESRGAN_x2plus.pth",
    model=model_esrgan,
    half=True,
    device="cuda",
)

print(f"Real-ESRGAN cargado. Upscaling {len(imagenes_generadas)} imagenes (1280x720 -> 1920x1080)...\n")
t0 = time.time()

for numero, img_path in imagenes_generadas.items():
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    output, _ = upsampler.enhance(img, outscale=1.5)
    cv2.imwrite(img_path, output)
    h, w = output.shape[:2]
    print(f"  escena_{numero:02d}.png -> {w}x{h}")

print(f"\nUpscale completo en {time.time()-t0:.1f}s")

In [ ]:
# Celda 5 - Liberar VRAM de FLUX.1 antes de cargar Wan 2.1
import gc

del pipe_img
torch.cuda.empty_cache()
gc.collect()
print(f"VRAM liberada. Disponible: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

In [ ]:
# Celda 6 - Generar videos con Wan 2.1 (25 steps, 41 frames, ~2.5s)

DEFAULT_VIDEO_NEGATIVE = "static, blurry, distorted"

videos_generados = {}

if escenas_video:
    print(f"Cargando Wan 2.1 para {len(escenas_video)} escenas con video IA...")

    try:
        from diffusers import WanImageToVideoPipeline
        from diffusers.utils import load_image

        pipe_video = WanImageToVideoPipeline.from_pretrained(
            "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers",
            torch_dtype=torch.float16,
        )
        pipe_video.enable_model_cpu_offload()

        t0_vid = time.time()

        for i, escena in enumerate(escenas_video):
            numero = escena["numero"]
            img_path = imagenes_generadas.get(numero)
            if not img_path:
                print(f"  [SKIP] Escena {numero}: no se genero imagen")
                continue

            print(f"[{i+1}/{len(escenas_video)}] Video escena {numero}...")

            image = load_image(img_path).resize((832, 480))

            neg = escena.get("negative_prompt") or DEFAULT_VIDEO_NEGATIVE
            if "static" not in neg.lower():
                neg = f"static, {neg}"

            output = pipe_video(
                image=image,
                prompt=escena["prompt"],
                negative_prompt=neg,
                num_inference_steps=25,
                guidance_scale=5.0,
                num_frames=41,
                generator=torch.Generator("cuda").manual_seed(SEED_BASE + numero),
            )

            video_path = OUTPUT_DIR / f"escena_{numero:02d}.mp4"

            from diffusers.utils import export_to_video
            export_to_video(output.frames[0], str(video_path), fps=16)

            videos_generados[numero] = str(video_path)
            print(f"  OK")
            torch.cuda.empty_cache()

        del pipe_video
        torch.cuda.empty_cache()
        gc.collect()
        print(f"\n{len(videos_generados)} videos IA en {(time.time()-t0_vid)/60:.1f} min")

    except Exception as e:
        print(f"Wan 2.1 no disponible ({e}), usando zoom/pan para todas las escenas")
        escenas_zoom = ESCENAS.copy()
        escenas_video = []
else:
    print("No hay escenas marcadas para video IA, todas usaran zoom/pan")

In [ ]:
# Celda 7 - Real-ESRGAN: upscale videos de 480P a 720P (frame por frame)

if videos_generados:
    print(f"Upscaling {len(videos_generados)} videos con Real-ESRGAN (480P -> 720P)...\n")
    t0 = time.time()

    for numero, video_path in videos_generados.items():
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 16
        frames = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            upscaled, _ = upsampler.enhance(frame, outscale=1.5)
            frames.append(upscaled)
        cap.release()

        if not frames:
            print(f"  escena_{numero:02d}.mp4 -> sin frames, saltando")
            continue

        h, w = frames[0].shape[:2]
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(video_path, fourcc, int(fps), (w, h))
        for frame in frames:
            writer.write(frame)
        writer.release()

        print(f"  escena_{numero:02d}.mp4 -> {w}x{h} ({len(frames)} frames)")

    print(f"\nUpscale video completo en {(time.time()-t0)/60:.1f} min")
else:
    print("No hay videos IA para upscale")

In [ ]:
# Celda 8 - Zoom/pan cinematico para escenas SIN video IA

ZOOM_DURACION_SEG = 5
ZOOM_FPS = 24
ZOOM_FACTOR = 1.15

MOVIMIENTOS = ["zoom_in", "zoom_out", "pan_left", "pan_right", "pan_up"]


def crear_video_zoom(img_path, output_path, movimiento="zoom_in",
                     duracion=ZOOM_DURACION_SEG, fps=ZOOM_FPS, zoom=ZOOM_FACTOR):
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    total_frames = duracion * fps

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (w, h))

    for frame_i in range(total_frames):
        t = frame_i / total_frames

        if movimiento == "zoom_in":
            s = 1.0 + (zoom - 1.0) * t
            cx, cy = w / 2, h / 2
        elif movimiento == "zoom_out":
            s = zoom - (zoom - 1.0) * t
            cx, cy = w / 2, h / 2
        elif movimiento == "pan_left":
            s = zoom
            cx = w / 2 + (w * 0.1) * (1 - t)
            cy = h / 2
        elif movimiento == "pan_right":
            s = zoom
            cx = w / 2 - (w * 0.1) * (1 - t)
            cy = h / 2
        elif movimiento == "pan_up":
            s = zoom
            cx = w / 2
            cy = h / 2 + (h * 0.08) * (1 - t)
        else:
            s = 1.0
            cx, cy = w / 2, h / 2

        new_w = int(w / s)
        new_h = int(h / s)
        x1 = max(0, int(cx - new_w / 2))
        y1 = max(0, int(cy - new_h / 2))
        x2 = min(w, x1 + new_w)
        y2 = min(h, y1 + new_h)

        crop = img[y1:y2, x1:x2]
        frame = cv2.resize(crop, (w, h), interpolation=cv2.INTER_LANCZOS4)
        writer.write(frame)

    writer.release()


escenas_sin_video = [e for e in ESCENAS if e["numero"] not in videos_generados]

print(f"Generando zoom/pan para {len(escenas_sin_video)} escenas...\n")

for i, escena in enumerate(escenas_sin_video):
    numero = escena["numero"]
    img_path = imagenes_generadas.get(numero)
    if not img_path:
        print(f"  [SKIP] Escena {numero}: no hay imagen")
        continue

    movimiento = MOVIMIENTOS[i % len(MOVIMIENTOS)]
    video_path = OUTPUT_DIR / f"escena_{numero:02d}.mp4"

    print(f"[{i+1}/{len(escenas_sin_video)}] Escena {numero} ({movimiento})...")
    crear_video_zoom(img_path, video_path, movimiento=movimiento)
    videos_generados[numero] = str(video_path)
    print(f"  OK")

print(f"\nTotal videos generados: {len(videos_generados)}")

In [ ]:
# Celda 9 - Resumen final
print("=" * 50)
print(f"PROYECTO: {PROYECTO_ID}")
print(f"=" * 50)

pngs = sorted(OUTPUT_DIR.glob("escena_*.png"))
mp4s = sorted(OUTPUT_DIR.glob("escena_*.mp4"))

print(f"\nImagenes generadas: {len(pngs)}")
for p in pngs:
    size_kb = p.stat().st_size / 1024
    img = cv2.imread(str(p))
    if img is not None:
        h, w = img.shape[:2]
        print(f"  {p.name} ({w}x{h}, {size_kb:.0f} KB)")
    else:
        print(f"  {p.name} ({size_kb:.0f} KB)")

print(f"\nVideos generados: {len(mp4s)}")
for p in mp4s:
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"  {p.name} ({size_mb:.1f} MB)")

total_mb = sum(p.stat().st_size for p in list(pngs) + list(mp4s)) / (1024 * 1024)
print(f"\nTamano total output: {total_mb:.1f} MB")
print("\nModelos usados:")
print("  - FLUX.1-schnell (imagenes 1280x720)")
print("  - Real-ESRGAN x2 (upscale imagenes a 1920x1080, video 480P a 720P)")
print("  - Wan 2.1 I2V 14B 480P (25 steps, 41 frames)")
print("\nListo para descarga automatica por el agente 3.2")